# 01 · User-Based Collaborative Filtering

**Role in the pipeline:** base scorer #1. Produces `scores_ubcf_{stack_train,val,test}` in the shared output format, which the stacker (`04_stack`) consumes.


| Baseline limitation | Fix here |
|---|---|
| Users not in the training matrix are skipped | **Fold-in**: a user's vector is built at scoring time from build-window clicks + the history they arrive with |
| Binary interactions, no time weighting | **Time decay** `exp(-age/τ)` on clicks (τ tuned on validation) |
| Popular articles create spurious similarity | **IUF weighting** `log(N_users / n_item_users)` |
| Users sharing 1 article look very similar | **Shrinkage** `sim · overlap / (overlap + λ)` |
| K = 200 untuned | K tuned on validation nDCG@10 |
| New articles silently dropped from evaluation | Scored as **unavailable** (`available = 0`), imputed with train mean, evaluated on full slates |
| Tie-breaking inflated MRR/nDCG | Shared tie-safe `evaluate()` |

**Leakage:** for each phase the matrix is built only from that phase's build days (see `common.PHASES`). Day 5 (stacker training rows) is scored by a model built on days 1–4.

In [1]:
import time
import numpy as np
import pandas as pd
from scipy import sparse

import common as C
import importlib
importlib.reload(C)

beh = C.load_behaviors()
cand = C.load_candidates(beh)
print(beh["split"].value_counts().to_dict(), f"| candidate rows: {len(cand):,}")

{'train': 126695, 'test': 73152, 'val': 30270} | candidate rows: 8,584,442


## Model

In [2]:
class UBCF:
    def __init__(self, k=100, shrink=10.0, tau_days=None, iuf=True, batch=256):
        self.k, self.shrink, self.tau_days, self.iuf, self.batch = k, shrink, tau_days, iuf, batch

    # ---- build the user-item matrix from the build-window click log ----
    def fit(self, log: pd.DataFrame):
        self.ref_time = log["time"].max()
        self.users = pd.Index(log["user_id"].unique())
        self.items = pd.Index(log["news_id"].unique())
        r = self.users.get_indexer(log["user_id"])
        c = self.items.get_indexer(log["news_id"])
        shape = (len(self.users), len(self.items))
        self.R_bin = sparse.csr_matrix((np.ones(len(log), np.float32), (r, c)), shape=shape)
        n_item_users = np.asarray(self.R_bin.sum(axis=0)).ravel()
        self.item_w = (np.log(len(self.users) / np.maximum(n_item_users, 1)).astype(np.float32)
                       if self.iuf else np.ones(len(self.items), np.float32))
        self.R_norm = self._weighted_normed(r, c, log["time"], shape)
        self.RnT = self.R_norm.T.tocsr()
        self.RbT = self.R_bin.T.tocsr()
        return self

    def _weighted_normed(self, r, c, times, shape):
        w = C.decay_weights(times, self.ref_time, self.tau_days) * self.item_w[c]
        M = sparse.csr_matrix((w, (r, c)), shape=shape)
        norms = np.sqrt(np.asarray(M.multiply(M).sum(axis=1)).ravel())
        return sparse.diags(1.0 / np.maximum(norms, 1e-12)) @ M

    # ---- score (user, candidate) pairs ----
    def score(self, qlog: pd.DataFrame, pairs: pd.DataFrame) -> pd.DataFrame:
        """qlog: clicks known per scoring user (build log + arrival history).
        pairs: rows with user_id, news_id. Returns ubcf_score, available and two
        trust features (neighbour strength, number of neighbours who clicked)."""
        qlog = qlog[qlog["news_id"].isin(self.items)]
        q_users = pd.Index(qlog["user_id"].unique())
        qr = q_users.get_indexer(qlog["user_id"])
        qc = self.items.get_indexer(qlog["news_id"])
        shape = (len(q_users), len(self.items))
        Qn = self._weighted_normed(qr, qc, qlog["time"], shape)
        Qb = sparse.csr_matrix((np.ones(len(qr), np.float32), (qr, qc)), shape=shape)
        self_idx = self.users.get_indexer(q_users)            # -1 if user not in build matrix

        n = len(pairs)
        score_ = np.full(n, np.nan, np.float32)
        strength_ = np.full(n, np.nan, np.float32)
        nclicks_ = np.full(n, np.nan, np.float32)
        pu = q_users.get_indexer(pairs["user_id"])             # -1 -> no clicks at all (cold user)
        pi = self.items.get_indexer(pairs["news_id"])          # -1 -> article never clicked (cold item)
        order = np.argsort(pu, kind="stable")                  # pairs grouped by query user
        pu_sorted = pu[order]
        k = min(self.k, len(self.users) - 1)

        for start in range(0, len(q_users), self.batch):
            end = min(start + self.batch, len(q_users))
            sims = (Qn[start:end] @ self.RnT).toarray()
            if self.shrink > 0:
                overlap = (Qb[start:end] @ self.RbT).toarray()
                sims *= overlap / (overlap + self.shrink)
            for j, s in enumerate(self_idx[start:end]):
                if s >= 0:
                    sims[j, s] = 0.0                          # exclude self
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            top_s = np.take_along_axis(sims, top, axis=1)
            rows = np.repeat(np.arange(end - start), k)
            W = sparse.csr_matrix((top_s.ravel(), (rows, top.ravel())), shape=(end - start, len(self.users)))
            Wb = sparse.csr_matrix(((top_s.ravel() > 0).astype(np.float32), (rows, top.ravel())), shape=W.shape)
            num = (W @ self.R_bin).toarray()
            cnt = (Wb @ self.R_bin).toarray()
            denom = np.abs(top_s).sum(axis=1)

            lo, hi = np.searchsorted(pu_sorted, [start, end])
            idx = order[lo:hi]                                 # pairs belonging to this batch
            lu = pu[idx] - start
            strength_[idx] = top_s.mean(axis=1)[lu]
            ok = pi[idx] >= 0
            idx, lu, it = idx[ok], lu[ok], pi[idx][ok]
            d = denom[lu]
            score_[idx] = np.where(d > 0, num[lu, it] / np.where(d > 0, d, 1), np.nan)
            nclicks_[idx] = cnt[lu, it]
        return pd.DataFrame(index=pairs.index, data={
            "ubcf_score": score_, "ubcf_strength": strength_, "ubcf_n_clicks": nclicks_,
            "available": (~np.isnan(score_)).astype(np.int8)})


def run_phase(phase, params, cand=cand, beh=beh):
    cfg = C.PHASES[phase]
    log = C.build_click_log(beh, cand, cfg["build_days"])
    c = C.phase_candidates(cand, phase)
    c = C.add_cold_start_flags(c, beh, log, cfg["score_days"])
    qlog = C.user_query_log(log, beh, cfg["score_days"], c["user_id"])
    model = UBCF(**params).fit(log)
    s = model.score(qlog, c[["user_id", "news_id"]])
    c = pd.concat([c, s], axis=1)
    c["score"] = c["ubcf_score"]
    return c

## Tune K, shrinkage, time decay on validation
Models are built on train days 1–5 and scored on the validation day. Unavailable pairs get the mean available score so every impression is ranked on its full slate (tie-safe).

In [3]:
GRID = [dict(k=k, shrink=s, tau_days=t)
        for k in (50, 100, 200) for s in (0.0, 10.0) for t in (None, 2.0)]

val_runs = []
for p in GRID:
    t0 = time.time()
    v = run_phase("val", p)
    fill = v.loc[v["available"] == 1, "score"].mean()
    v["score"] = v["score"].fillna(fill)
    r = C.evaluate(v, by_segment=False).iloc[0]
    val_runs.append({**p, **r.to_dict(), "secs": round(time.time() - t0, 1)})
    print(p, f"nDCG@10={r['nDCG@10']:.4f}  AUC={r['AUC']:.4f}")

tuning = pd.DataFrame(val_runs).sort_values("nDCG@10", ascending=False)
BEST = {k: (None if pd.isna(v) else v) for k, v in tuning.iloc[0][["k", "shrink", "tau_days"]].items()}
BEST["k"] = int(BEST["k"])
print("best:", BEST)
tuning.head(10)

{'k': 50, 'shrink': 0.0, 'tau_days': None} nDCG@10=0.2794  AUC=0.5214


{'k': 50, 'shrink': 0.0, 'tau_days': 2.0} nDCG@10=0.2730  AUC=0.5161


{'k': 50, 'shrink': 10.0, 'tau_days': None} nDCG@10=0.2837  AUC=0.5263


{'k': 50, 'shrink': 10.0, 'tau_days': 2.0} nDCG@10=0.2768  AUC=0.5199


{'k': 100, 'shrink': 0.0, 'tau_days': None} nDCG@10=0.2818  AUC=0.5244


{'k': 100, 'shrink': 0.0, 'tau_days': 2.0} nDCG@10=0.2750  AUC=0.5178


... (3 outputs left out)

{'k': 200, 'shrink': 0.0, 'tau_days': 2.0} nDCG@10=0.2765  AUC=0.5193


{'k': 200, 'shrink': 10.0, 'tau_days': None} nDCG@10=0.2861  AUC=0.5293


{'k': 200, 'shrink': 10.0, 'tau_days': 2.0} nDCG@10=0.2797  AUC=0.5220
best: {'k': 200, 'shrink': 10.0, 'tau_days': None}


,k,shrink,tau_days,impressions,MRR,nDCG@5,nDCG@10,AUC,secs
10,200,10.0,NaN,30270.0,0.254192,0.227382,0.286090,0.529291,40.4
6,100,10.0,NaN,30270.0,0.253922,0.226808,0.285626,0.529091,36.5
8,200,0.0,NaN,30270.0,0.252030,0.225222,0.284192,0.527377,28.4
2,50,10.0,NaN,30270.0,0.252053,0.224689,0.283727,0.526307,35.6
4,100,0.0,NaN,30270.0,0.250389,0.222732,0.281816,0.524377,26.4
11,200,10.0,2.0,30270.0,0.247432,0.220854,0.279697,0.521996,273.1
0,50,0.0,NaN,30270.0,0.248086,0.220826,0.279359,0.521428,24.8
7,100,10.0,2.0,30270.0,0.246051,0.219688,0.278331,0.521232,34.4
3,50,10.0,2.0,30270.0,0.244887,0.218171,0.276782,0.519882,33.6
9,200,0.0,2.0,30270.0,0.244600,0.217938,0.276539,0.519287,28.8


## Score all phases with the best setting and save (shared output format)

In [4]:
EXTRA = ["ubcf_strength", "ubcf_n_clicks"]
for phase in C.PHASES:
    df = run_phase(phase, BEST)
    path = C.save_scores(df, "ubcf", phase, extra_cols=EXTRA)
    print(f"{phase:12s} rows={len(df):,}  available={df['available'].mean():.1%}  "
          f"cold users={df.groupby('imp_id')['user_cold'].first().mean():.1%}  "
          f"cold items={df['item_cold'].mean():.1%}  -> {path}")

stack_train  rows=1,231,673  available=47.0%  cold users=0.6%  cold items=52.7%  -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_ubcf_stack_train.parquet


val          rows=1,222,429  available=42.9%  cold users=0.5%  cold items=56.9%  -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_ubcf_val.parquet


test         rows=2,740,998  available=17.6%  cold users=2.7%  cold items=81.9%  -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_ubcf_test.parquet


## Standalone evaluation (val & test, by cold-start segment)

In [5]:
res = C.standalone_eval("ubcf")
for ph, table in res.items():
    table = table[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
    print(f"\n=== UBCF · {ph} ===")
    print(table.round(4).to_string())
    table.to_csv(f"{C.OUT_DIR}/metrics_ubcf_{ph}.csv")


=== UBCF · val ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          30270  0.2546  0.2276   0.2866  0.5304
Cold users (0 history)                         772  0.2572  0.2347   0.2841  0.5261
Sparse users (1-4 history)                    3247  0.2685  0.2517   0.3144  0.5211
Warm users (>=5 history)                     26251  0.2528  0.2245   0.2832  0.5317
Clicked article seen before                   8276  0.2148  0.1905   0.2255  0.3750
Clicked article new (no prior clicks)        21994  0.2696  0.2416   0.3096  0.5889

=== UBCF · test ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          73152  0.2486  0.2300   0.2931  0.5123
Cold users (0 history)                        2214  0.2555  0.2332   0.2937  0.5004
Sparse users (1-4 history)                    8092  0.2713  0.2693   0.3314  0.4921
Warm users (>=5 history)           